In [ ]:
from tokenizer.tokenizer import encode, decode
import json
from collections import Counter

In [ ]:
with open("tokenizer/merges.json", "r") as f:
    merges = json.load(f)

In [ ]:
curr_vocab_length = merges[-1]["z"]
curr_vocab_length

In [ ]:
spl_tokens = ["<eos>", "<bos>", "<pad>", "<system>", "<user>", "<sep>"]

In [ ]:
vocab_dict = {}

for part in spl_tokens:
    try:
        vocab_dict[part]["freq"] += 1
    except KeyError:
        vocab_dict[part] = {"freq":1, "token_ids": encode(part)}
        
print("Total Words", len(vocab_dict))
# print(vocab_dict)


def update_list(token_ids, x, y, curr_vocab_length):
    new_list = []
    i = 0
    
    while i < len(token_ids):
        if token_ids[i:i+2] == [x,y]:
            new_list.append(curr_vocab_length)
            i += 2
        else:
            new_list.append(token_ids[i])
            i += 1
            
    return new_list


while max([len(encode(token, merges)) for token in spl_tokens]) > 1:
    freq_counter = Counter()
    for value in vocab_dict.values():
        token_ids = value["token_ids"]
        list_len = len(token_ids)
        
        if list_len >= 2:
            for i in range(list_len-1):
                freq_counter[(token_ids[i], token_ids[i+1])] += value["freq"]
                
    (x,y), freq = freq_counter.most_common(1)[0]

    curr_vocab_length += 1
    print("x: ", x, "  y: ", y, "  freq: ", freq, "  z:", curr_vocab_length)

    for val in vocab_dict.values():
        val["token_ids"] = update_list(val["token_ids"], x, y, curr_vocab_length)

    merges.append({"x": x, "y": y, "z": curr_vocab_length})

with open("merges.json", "w") as f:
    json.dump(merges, f)
